In [0]:
# Create two DataFrames
data1 = [(1, "Alice"), (2, "Bob"), (3, "Charlie")]
df1 = spark.createDataFrame(data1, ["id", "name"])

data2 = [(1, "HR"), (2, "Engineering"), (4, "Marketing")]
df2 = spark.createDataFrame(data2, ["id", "department"])

# Join DataFrames on 'id'
joined_df = df1.join(df2, "id")

display(joined_df)

joined_df.explain()

id,name,department
1,Alice,HR
2,Bob,Engineering


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   Project [id#124L, name#125, department#145]
   +- BroadcastHashJoin [id#124L], [id#144L], Inner, BuildRight, false, true
      :- Filter isnotnull(id#124L)
      :  +- LocalTableScan [id#124L, name#125]
      +- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=158]
         +- Filter isnotnull(id#144L)
            +- LocalTableScan [id#144L, department#145]




# 1. Sort Merge Join

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'-1'

In [0]:
from pyspark.sql.functions import broadcast

# Small DataFrame with increased records
small_data = [(i, f"category_{i}") for i in range(1, 1000000)]
small_df = spark.createDataFrame(small_data, ["id", "category"])

# Big DataFrame
big_data = [(i, f"value_{i}") for i in range(1, 100000)]
big_df = spark.createDataFrame(big_data, ["id", "value"])

# Sort Merge join
joined_df_smj = big_df.join(small_df, "id", "left_outer")

display(joined_df_smj)

id,value,category
6,value_6,category_6
7,value_7,category_7
19,value_19,category_19
22,value_22,category_22
25,value_25,category_25
26,value_26,category_26
29,value_29,category_29
31,value_31,category_31
32,value_32,category_32
34,value_34,category_34


# 2. Broadcast Join Execution

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "30mb")

spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'30mb'

In [0]:
from pyspark.sql.functions import broadcast

# Small DataFrame
small_data = [(1, "A"), (2, "B"), (3, "C")]
small_df_1 = spark.createDataFrame(small_data, ["id", "category"])

# Big DataFrame
big_data = [(i, f"value_{i}") for i in range(1, 1000001)]
big_df_1 = spark.createDataFrame(big_data, ["id", "value"])

# Broadcast join
joined_df_1 = big_df_1.join(broadcast(small_df_1), "id")

display(joined_df_1)

id,value,category
1,value_1,A
2,value_2,B
3,value_3,C


# 3. Caching performance


In [0]:
from pyspark.sql.functions import col
import time

# Generate a larger DataFrame for performance testing
data = [(i, i % 100, f"text_{i}") for i in range(1, 1000001)]
df = spark.createDataFrame(data, ["id", "group", "text"])

# Define a costly aggregation
def run_aggregation(df):
    return df.groupBy("group").count()

# Measure time without caching
start = time.time()
result_no_cache = run_aggregation(df)
result_no_cache.collect()
no_cache_time = time.time() - start

print("first time: " + str(no_cache_time))

result_no_cache = run_aggregation(df)
result_no_cache.collect()
no_cache_time = time.time() - start

print("second time: " + str(no_cache_time))

first time: 8.209133386611938
second time: 13.118645191192627


In [0]:
# Cache the DataFrame
df.cache()
df.count()  # Materialize cache

# Measure time with caching
start = time.time()
result_cache = run_aggregation(df)
result_cache.collect()
cache_time = time.time() - start

print("first time: " +str(cache_time))

# Measure time with caching
start = time.time()
result_cache = run_aggregation(df)
result_cache.collect()
cache_time = time.time() - start

print("second time: " +str(cache_time))


first time: 2.3898143768310547
second time: 2.6517415046691895


In [0]:
from pyspark import StorageLevel

# MEMORY_ONLY (default)
df.cache()
df.count()



In [0]:
# MEMORY_AND_DISK
from pyspark import StorageLevel
df.persist(StorageLevel.MEMORY_AND_DISK)
df.count()



In [0]:
df.unpersist()

DataFrame[id: bigint, group: bigint, text: string]

In [0]:
# MEMORY_ONLY_SER
df.persist(StorageLevel.MEMORY_AND_DISK_DESER)
df.count()



In [0]:
from pyspark import StorageLevel

# MEMORY_AND_DISK_SER
from pyspark import StorageLevel
df.persist(StorageLevel.MEMORY_AND_DISK_2)
df.count()



1000000

In [0]:
# DISK_ONLY
df.persist(StorageLevel.DISK_ONLY)
df.count()

## Spark Caching & Persistence Levels

Spark provides several persistence levels for caching DataFrames/RDDs:

| Level                | Storage         | Uses & Advantages                          | Disadvantages                |
|----------------------|----------------|--------------------------------------------|------------------------------|
| MEMORY_ONLY          | RAM            | Fast access, avoids disk I/O               | May evict partitions if RAM is full |
| MEMORY_AND_DISK      | RAM + Disk     | Handles large data, spills to disk         | Slower if disk is used       |
| MEMORY_AND_DISK_DESER  | RAM+Disk (deserialized)|fast access                       | Higher Memory                |
| DISK_ONLY            | Disk           | Works for very large data                  | Slowest, disk I/O            |
| OFF_HEAP             | Off-heap RAM   | Avoids JVM GC, used for Tungsten           | Requires config, less common |

**Choose the level based on data size and available resources.**


# 4. Data Skew Problem

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.shuffle.partitions", 20)

print(spark.conf.get("spark.sql.adaptive.enabled"))
print(spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))
print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
print(spark.conf.get("spark.sql.shuffle.partitions"))

false
false
-1
20


In [0]:
# Create a highly skewed DataFrame with 10 unique keys and lots of duplicate values
skew_data = [(1, f"skewed_{i}") for i in range(1, 100001)] + \
            [(k, f"normal_{j}") for k in range(2, 11) for j in range(1, 100)]
skew_df = spark.createDataFrame(skew_data, ["key", "value"])

# Small DataFrame for join with 10 unique keys
small_data = [(k, f"cat_{k}") for k in range(1, 11000)]
small_df = spark.createDataFrame(small_data, ["key", "category"])

skew_df.cache()
small_df.cache()

display(skew_df.groupBy("key").count())
display(small_df.groupBy("key").count())

# Join operation likely to cause long processing time due to skew

key,count
4,99
6,99
3,99
2,99
5,99
1,100000
9,99
7,99
8,99
10,99


key,count
4,1
26,1
29,1
61,1
92,1
163,1
171,1
229,1
230,1
276,1


In [0]:
union_skew_df = skew_df
union_small_df = small_df
for _ in range(9):
    union_skew_df = union_skew_df.union(skew_df)
    union_small_df = union_small_df.union(small_df)

union_skew_df.cache()
union_small_df.cache()

display(union_skew_df.groupBy("key").count())
display(union_small_df.groupBy("key").count())

key,count
4,990
6,990
3,990
2,990
5,990
1,1000000
9,990
7,990
8,990
10,990


key,count
4,10
26,10
29,10
61,10
92,10
163,10
171,10
229,10
230,10
276,10


In [0]:
display(union_skew_df.groupBy("key").count())
display(union_small_df.groupBy("key").count())

key,count
4,990
6,990
3,990
2,990
5,990
1,1000000
9,990
7,990
8,990
10,990


key,count
4,10
26,10
29,10
61,10
92,10
163,10
171,10
229,10
230,10
276,10


In [0]:
skew_df_1 = union_skew_df.repartition(3, "key")
small_df_1 = union_small_df.repartition(3, "key")

joined_skew_df = skew_df_1.join(small_df_1, "key", "left_outer")

display(joined_skew_df.count())

10089100

# 5. Salting Approach to solve Data Skew

In [0]:
# Enable AQE and skew join handling
#spark.conf.set("spark.sql.adaptive.enabled", True)
spark.conf.set("spark.sql.shuffle.partitions", 30)
#spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
#spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "3MB")
#spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes","8MB")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$shutdown$4(ExecContextState.scala:462)
	at scala.collection.immutable.List.flatMap(List.scala:366)
	at com.databricks.spark.chauffeur.ExecContextState.shutdown(ExecContextState.scala:462)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.shutdownContext(ExecutionContextManagerV1.scala:768)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.$anonfun$clearAllRunningState$1(ExecutionContextManagerV1.scala:724)
	at scala.collection.immutable.List.foreach(List.scala:431)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.clearAllRunningState(ExecutionContextManagerV1.scala:722)
	at com.databricks.spark.chauffeur.ChauffeurState.clearAllRunningState(ChauffeurState.scala:516)
	at com.databricks.spark.chauffeur.ChauffeurState.processDriverStateChange(ChauffeurState.scala:319)
	at com.databricks.spark.chauffeur.Chauffeur.onDriverStateChange(Chauf

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, expr, concat_ws

# Add salt column and append to key in skew_df_1
num_salts = 20
skew_df_1_salted = skew_df_1.withColumn("salt", (monotonically_increasing_id() % num_salts))
skew_df_1_salted = skew_df_1_salted.withColumn("salted_key", concat_ws("_", skew_df_1_salted["key"], skew_df_1_salted["salt"]))

skew_df_1_salted.filter("key == 1").show(truncate=False)

# Add salt column and append to key in small_df_1 (only one salt value per key)
small_df_1_salted = small_df_1.withColumn("salt", expr("0"))
small_df_1_salted = small_df_1_salted.withColumn("salted_key", concat_ws("_", small_df_1_salted["key"], small_df_1_salted["salt"]))

small_df_1_salted.filter("key == 1").show(truncate=False)

# Join on salted_key
joined_skew_df = skew_df_1_salted.join(small_df_1_salted, "salted_key", "inner")

display(joined_skew_df.select("*").count())

+---+---------+----+----------+
|key|value    |salt|salted_key|
+---+---------+----+----------+
|1  |skewed_1 |4   |1_4       |
|1  |skewed_2 |5   |1_5       |
|1  |skewed_3 |6   |1_6       |
|1  |skewed_4 |7   |1_7       |
|1  |skewed_5 |8   |1_8       |
|1  |skewed_6 |9   |1_9       |
|1  |skewed_7 |10  |1_10      |
|1  |skewed_8 |11  |1_11      |
|1  |skewed_9 |12  |1_12      |
|1  |skewed_10|13  |1_13      |
|1  |skewed_11|14  |1_14      |
|1  |skewed_12|15  |1_15      |
|1  |skewed_13|16  |1_16      |
|1  |skewed_14|17  |1_17      |
|1  |skewed_15|18  |1_18      |
|1  |skewed_16|19  |1_19      |
|1  |skewed_17|0   |1_0       |
|1  |skewed_18|1   |1_1       |
|1  |skewed_19|2   |1_2       |
|1  |skewed_20|3   |1_3       |
+---+---------+----+----------+
only showing top 20 rows
+---+--------+----+----------+
|key|category|salt|salted_key|
+---+--------+----+----------+
|1  |cat_1   |0   |1_0       |
|1  |cat_1   |0   |1_0       |
|1  |cat_1   |0   |1_0       |
|1  |cat_1   |0   |1_

504460